In [1]:
# ===== EXPERIMENT CONFIGURATION =====

PROJECT_NAME = "Road Traffic Detection using Drone (ML)"
MODEL_NAME = "yolov11n"

DATASET_YAML = r"D:\Road Traffic Detection using Drone (ML)\Drone Datasets\dataset.yaml"
DATASET_ROOT = r"D:\Road Traffic Detection using Drone (ML)\Drone Datasets\FINAL_DATASET_PROCESSED"

OUTPUT_DIR = r"D:\Road Traffic Detection using Drone (ML)\Outputs-Results"
RUN_NAME = "yolov11n"

IMG_SIZE = 640
EPOCHS = 35
BATCH = 4
WORKERS = 4
SEED = 42
PATIENCE = 10

CACHE = True
DEVICE = 0

In [2]:
from ultralytics import YOLO
import torch
import os
import pandas as pd

In [3]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory Total (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("WARNING: GPU not detected. Training will be slow.")

Torch version: 2.5.1+cu121
CUDA available: True
CUDA version: 12.1
GPU: NVIDIA GeForce RTX 2050
GPU Memory Total (GB): 4.29


In [4]:
print("Dataset YAML exists:", os.path.exists(DATASET_YAML))
print("Dataset root exists:", os.path.exists(DATASET_ROOT))

Dataset YAML exists: True
Dataset root exists: True


In [6]:
train_images = os.listdir(os.path.join(DATASET_ROOT, "images/train"))
train_labels = os.listdir(os.path.join(DATASET_ROOT, "labels/train"))

val_images = os.listdir(os.path.join(DATASET_ROOT, "images/val"))
val_labels = os.listdir(os.path.join(DATASET_ROOT, "labels/val"))

print("Train Images:", len(train_images))
print("Train Labels:", len(train_labels))

print("Val Images:", len(val_images))
print("Val Labels:", len(val_labels))

Train Images: 20000
Train Labels: 20000
Val Images: 5000
Val Labels: 5000


In [7]:
model = YOLO("yolo11n.pt")

In [8]:
CHECKPOINT_PATH = os.path.join(
    OUTPUT_DIR,
    RUN_NAME,
    "weights",
    "last.pt"
)

print("Checkpoint path:", CHECKPOINT_PATH)

resume_training = False

if os.path.exists(CHECKPOINT_PATH):
    print("Checkpoint found → Resuming training")
    resume_training = True
else:
    print("No checkpoint found → Starting fresh training")

Checkpoint path: D:\Road Traffic Detection using Drone (ML)\Outputs-Results\yolov11n\weights\last.pt
No checkpoint found → Starting fresh training


In [9]:
train_params = dict(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    seed=SEED,
    patience=PATIENCE,
    cache=CACHE,
    amp=True,
    project=OUTPUT_DIR,
    name=RUN_NAME,
    exist_ok=True
)

# Resume logic
if resume_training:
    train_params["resume"] = CHECKPOINT_PATH

In [11]:
print("RUN_NAME =", RUN_NAME)

RUN_NAME = yolov11n


In [12]:
if resume_training:
    model = YOLO(CHECKPOINT_PATH)

    results = model.train(
        resume=True,
        **train_params
    )
else:
    results = model.train(
        **train_params
    )

New https://pypi.org/project/ultralytics/8.4.28 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.22  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Road Traffic Detection using Drone (ML)\Drone Datasets\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=35, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mo

In [13]:
RESULTS_FILE = os.path.join(OUTPUT_DIR, RUN_NAME, "results.csv")

if os.path.exists(RESULTS_FILE):
    df = pd.read_csv(RESULTS_FILE)
    display(df.tail())
else:
    print("Results file not found.")

,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2,lr/pg3,lr/pg4,lr/pg5,lr/pg6,lr/pg7
30,31,25936.2,0.99732,0.56104,0.84799,0.70406,0.62084,0.65509,0.46194,1.00020,0.60407,0.87173,0.004543,0.001514,0.004543,0.001514,0.004543,0.001514,0.004543,0.001514
31,32,26627.0,0.98623,0.55394,0.84643,0.74186,0.61097,0.65621,0.46313,0.99674,0.59974,0.87102,0.003694,0.001231,0.003694,0.001231,0.003694,0.001231,0.003694,0.001231
32,33,27309.4,0.97925,0.54682,0.84506,0.75331,0.60331,0.65851,0.46580,0.99417,0.59669,0.87071,0.002846,0.000949,0.002846,0.000949,0.002846,0.000949,0.002846,0.000949
33,34,27989.9,0.96818,0.53796,0.84307,0.73616,0.60907,0.65898,0.46662,0.99190,0.59437,0.87001,0.001997,0.000666,0.001997,0.000666,0.001997,0.000666,0.001997,0.000666
34,35,28682.5,0.96009,0.53191,0.84143,0.74778,0.60413,0.66024,0.46803,0.98976,0.59266,0.86946,0.001149,0.000383,0.001149,0.000383,0.001149,0.000383,0.001149,0.000383


In [14]:
BEST_MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    RUN_NAME,
    "weights",
    "best.pt"
)

print("Best model path:", BEST_MODEL_PATH)
print("Exists:", os.path.exists(BEST_MODEL_PATH))

Best model path: D:\Road Traffic Detection using Drone (ML)\Outputs-Results\yolov11n\weights\best.pt
Exists: True


In [15]:
model = YOLO(BEST_MODEL_PATH)
metrics = model.val()

metrics

Ultralytics 8.4.22  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 953.8316.9 MB/s, size: 174.7 KB)
val: Scanning D:\Road Traffic Detection using Drone (ML)\Drone Datasets\FINAL_DATASET_PROCESSED\labels\val.cache... 5000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5000/5000  0.0s
val: D:\Road Traffic Detection using Drone (ML)\Drone Datasets\FINAL_DATASET_PROCESSED\images\val\UAVDT_YOLO_M0606_img001213.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 313/313 6.3it/s 49.4s<0.2s
                   all       5000      79843      0.748      0.604      0.661      0.469
                   car        888       7244      0.806      0.614      0.705      0.465
                   bus       3832      63461      0.928      0.956      0.977      

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001E5B260B310>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
 

In [16]:
size_mb = os.path.getsize(BEST_MODEL_PATH) / (1024 * 1024)
print("Model size (MB):", round(size_mb, 2))

Model size (MB): 5.21
